In [4]:
# !pip -q install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 80.4 MB/s eta 0:00:00


In [10]:
# -- IMPORTS --
from pathlib import Path
import os
import json
import re
import fitz
import glob

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base = Path("/content/drive/MyDrive/geant_chatbot")

raw_ZENODO = base / "raw_docs" / "zenodo_docs"
raw_CONNECT = base / "raw_docs" / "connect_articles"
out_docs = base / "processed" / "docs_json"

out_docs.mkdir(parents=True, exist_ok=True)

print("Zenodo PDFs:", len(list(raw_ZENODO.glob("*.pdf"))))
print("Connect articles:", len(list(raw_CONNECT.glob("*.txt"))))
print("Output folder:", out_docs)

Zenodo PDFs: 10
Connect articles: 46
Output folder: /content/drive/MyDrive/geant_chatbot/processed/docs_json


In [6]:
def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_pdf_text(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    parts = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        if text and text.strip():
            parts.append(f"[PAGE {i+1}] {text}")
    return clean_text("\n".join(parts))

def save_doc(doc_id: str, source_path: str, doc_type: str, text: str):
    record = {
        "doc_id": doc_id,
        "doc_type": doc_type,
        "source_path": source_path,
        "text": text}

    out_path = out_docs / f"{doc_id}.json"
    out_path.write_text(
        json.dumps(record, ensure_ascii=False, indent=2),
        encoding="utf-8")

In [11]:
count = 0

for pdf in raw_ZENODO.glob("*.pdf"):
    text = extract_pdf_text(pdf)
    if len(text) < 500:
        print("Skipping short PDF:", pdf.name)
        continue

    doc_id = "pdf__" + pdf.stem[:150]
    save_doc(
        doc_id=doc_id,
        source_path=str(pdf),
        doc_type="pdf",
        text=text)

    count += 1
    print("Extracted PDF:", pdf.name)

print("PDF extraction done. Total:", count)


Extracted PDF: GN5-2 Intelligent Networks The Rise of Generative AI in Network Management_GN5-2-White-Paper_Gen-AI-in-Network-Management.pdf
Extracted PDF: GN5-2 WFO Telemetry Module Incubator Project Closure Report_GN5-1_WFO-Telemetry-Module-Incubator-Project-Closure-Report_Public.pdf
Extracted PDF: GN5-2 D32 Compendium Report_GN5-2_D3.2_Compendium-Report.pdf
Extracted PDF: GN5-2 D21 Project Communications Strategy and Plan_GN5-2_D2-1_Project-Communications-Strategy-and-Plan.pdf
Extracted PDF: GN5-2 D95 Open Source and Licence Support Report_GN5-2_D9.5_Open-Source-and-Licence-Support-Report.pdf
Extracted PDF: GN5-2 Fibre Sensing Technologies Users and Use Cases for NRENs_GN5-2-White-Paper_Fibre-Sensing-Technologies.pdf
Extracted PDF: Switch OAV Architecture Analysis_GN5-2_White-Paper_Switch-OAV-Architecture-Analysis.pdf
Extracted PDF: GN5-2 MARnet OAV Architecture Analysis_GN5-2-White-Paper_MARnet-OAV-Architecture-Analysis.pdf
Extracted PDF: GN5-2 Towards Quantum-Safe Networking_GN5-2

In [12]:
count = 0

for txt in raw_CONNECT.glob("*.txt"):
    text = clean_text(txt.read_text(encoding="utf-8", errors="ignore"))
    if len(text) < 200:
        print("Skipping short TXT:", txt.name)
        continue

    doc_id = "txt__" + txt.stem
    save_doc(
        doc_id=doc_id,
        source_path=str(txt),
        doc_type="txt",
        text=text
    )
    count += 1

print("TXT extraction done. Total:", count)


TXT extraction done. Total: 46


In [13]:
files = sorted(glob.glob(str(out_docs / "*.json")))
print("Total JSON docs:", len(files))

sample = files[0]
print("Sample file:", sample)

doc = json.loads(open(sample, "r", encoding="utf-8").read())
print("doc_id:", doc["doc_id"])
print("doc_type:", doc["doc_type"])
print("text preview:\n", doc["text"][:600])


Total JSON docs: 56
Sample file: /content/drive/MyDrive/geant_chatbot/processed/docs_json/pdf__GN5-2 D21 Project Communications Strategy and Plan_GN5-2_D2-1_Project-Communications-Strategy-and-Plan.json
doc_id: pdf__GN5-2 D21 Project Communications Strategy and Plan_GN5-2_D2-1_Project-Communications-Strategy-and-Plan
doc_type: pdf
text preview:
 [PAGE 1] © GÉANT Association on behalf of the GN5-2 project. The research leading to these results has received funding from the European Union’s Horizon Europe research and innovation programme under Grant Agreement No. 101194278 (GN5-2). Co-funded by the European Union. Views and opinions expressed are however those of the author(s) only and do not necessarily reflect those of the European Union. The European Union cannot be held responsible for them. 21-03-2025 Deliverable GN5-2 D2.1 Project Communications Strategy and Plan Contractual Date: 31-03-2025 Actual Date: 21-03-2025 Grant Agreemen
